# Seasonal SMME Model Selection and Data Generation
This notebook walks through the selection process and data generation for our Skilled Multi Model Ensemble (SMME) for our regions of interest. This file assumes that you have generated the metrics for all models, regions, and seasons of interest with seasonal_metrics_calculation.ipynb

## Criteria for a skillful model
- (Potential Skill is greater than 0.3) OR (AN and BN tercile hitrate are above 0.4) for that region at the specified lead category if applicable

For each region, four separate output files are generated, corresponding to general, short, medium, and long lead time categories. Each file contains the average predicted precipitation of SMME for that lead category.

Within a given lead time category, models are first evaluated for skill. If the average metric (e.g., potential skill) for that category exceeds the skill threshold (> 0.3), the model is considered skillful for that region and season at that lead category. Once all skillful models are identified, their precipitation predictions are averaged to produce the SMME estimate for that region and season.

For example, at the medium lead time for Eastern East Africa (EEA) during the MAM season, if GEM5 and CanESM both exceed the skill threshold, the SMME medium lead time model's predicted precipitation for EEA MAM is computed as the average of those two models' predicted precipitation.

Thus, each regional SMME output (e.g., for EEA) includes four files—SMME general, short, medium, and long—where each file summarizes the average precipitation across all skillful models, separately for each season (e.g., OND and MAM).

----------------------------------------

Change the paths underneath the import statement to your paths. These include:
- A path to the computed skill metrics (Potential Skill/Unconditional Bias/Skill Score/Conditional Bias)
- A path to AN hitrate metrics data
- A path to BN hitrate metrics data
- A save folder path for saving the text files describing SMME models for each region and season
- A save folder path for saving the raw SMME seasonal data (try to have this path as the same path as the seasonal data for the rest of the models. This facilitates efficient plotting)

In [1]:
# import necessary packages
import xarray as xr
import numpy as np
import pandas as pd
import os

In [34]:
'''Change these paths to your paths
'''

# path to skill metrics data
skill_metrics_data = 'data/csv/metrics/skill_metrics.csv'

# path to tercile metrics data (AN and BN)
AN_hitrate_data = 'data/csv/metrics/AN_hitrate_metrics.csv'
BN_hitrate_data = 'data/csv/metrics/BN_hitrate_metrics.csv'

# path to folder for saving text files for SMME models selected
SMME_text_save_path = 'data/SMME_text_files/'

# path to folder for saving raw seasonal SMME data
seasonal_data_save_path = 'data/seasonal/csv/'

In [19]:
# potential skill, an and bn cutoffs for SMME selection
potential_skill_cutoff = 0.3
an_cutoff = 0.4
bn_cutoff = 0.4

In [37]:
def keep_months_of_prediction(df, category):
    '''Helper function to keep the months of prediction
    for each model for each region and season.
    '''
    temp = df.copy().dropna()
    # keep months of prediction of each model for each region and season
    if category == 'short':
        temp = temp.loc[temp['lead_time'] < 2]
    elif category == 'medium':
        temp = temp.loc[(temp['lead_time'] >= 2) & (temp['lead_time'] < 4)]
    elif category == 'long':
        temp = temp.loc[temp['lead_time'] >= 4]
    else:
        pass
    return temp.dropna()

In [38]:
# import skill metrics data (potential skill, AN and BN)
potential_skill = pd.read_csv(skill_metrics_data, index_col=False)
potential_skill = potential_skill.drop(columns = ['conditional_bias', 'unconditional_bias', 'skill_score'])
an = pd.read_csv(AN_hitrate_data, index_col=False)
bn = pd.read_csv(BN_hitrate_data, index_col=False)

# Remove rows where 'model' contains 'MME' or 'SMME' in any form
pattern = 'MME|SMME'

potential_skill = potential_skill.loc[~potential_skill['model'].str.contains(pattern, case=False, na=False)]
an = an.loc[~an['model'].str.contains(pattern, case=False, na=False)]
bn = bn.loc[~bn['model'].str.contains(pattern, case=False, na=False)]

# remaming columns for clarity
an = an.rename(columns = {'agreement': 'an_agreement'})
bn = bn.rename(columns = {'agreement': 'bn_agreement'})

# merging an and bn to work with one dataframe
an_bn = an.merge(bn, on=['region', 'model', 'season', 'month_of_prediction'], how='inner')

In [ ]:
# load in raw seasonal data for all models and regions
# this is used to compute average precip for seasonal SMME later on

dfs_dict = {}

# List all CSV files in the target directory
list_of_files = os.listdir(seasonal_data_save_path)

# Build full file paths for seasonal CSV files
files_path = [os.path.join(seasonal_data_save_path, f) for f in list_of_files if f.endswith('_seasonal.csv')]

# Read each CSV and store in the dictionary
for f in files_path:
    df = pd.read_csv(f)
    dfs_dict[f] = df

# Merge all dataframes into one
merged_csv_data = pd.concat(dfs_dict.values(), ignore_index=True)


# Creating the General seasonal SMME model, using all 7 months of metrics and averaging them out

In [49]:
# keeping all months
potential_skill_general = keep_months_of_prediction(potential_skill, 'all').drop(columns=['lead_time'])

# merging potential skill with an and bn
merged_metrics_general = potential_skill_general.merge(an_bn, on=['region', 'model', 'season', 'month_of_prediction'], how='inner')

# taking the mean of the month of prediction for each region and model and season
merged_metrics_general = merged_metrics_general.groupby(['region', 'model', 'season'])[['potential_skill', 'an_agreement', 'bn_agreement']].mean().reset_index().dropna()

# keep models with potential skill > potential_skill_cutoff or (an_agreement > an_cutoff and bn_agreement > bn_cutoff)
merged_metrics_general = merged_metrics_general[(merged_metrics_general['potential_skill'] > potential_skill_cutoff) |
                                                ((merged_metrics_general['an_agreement'] > an_cutoff) & (merged_metrics_general['bn_agreement'] > bn_cutoff))]
merged_metrics_general['region_season'] = merged_metrics_general['region'] + " | " + merged_metrics_general['season'] # compile region and season into one column, split by " | "
merged_metrics_general = merged_metrics_general.drop(columns = ['region', 'season'])

In [50]:
# create a nested dictionary where the keys are the regions, the values are lists of models
models_general = merged_metrics_general.groupby('region_season')['model'].apply(list).to_dict()
models_general

# separate the region and season into nested keys
SMME_models_general = {}
SMME_models_general['metrics'] = {
    'potential_skill': potential_skill_cutoff,
    'an_hitrate': an_cutoff,
    'bn_hitrate': bn_cutoff,
    'SMME_criteria': f'(potential_skill > {potential_skill_cutoff}) OR (an AND bn) > {an_cutoff}'
}
for region_season, model_list in models_general.items():
    region, season = region_season.split(" | ")
    if region not in SMME_models_general:
        SMME_models_general[region] = {}
    SMME_models_general[region][season] = model_list

# create a text file of the SMME models chosen
with open(os.path.join(SMME_text_save_path, 'SMME_models_general.txt'), 'w') as f:
    f.write("General SMME Models\n")
    for region, seasons in SMME_models_general.items():
        f.write(f"{region}:\n")
        for season, models in seasons.items():
            f.write(f"  {season}: {models}\n")

In [51]:
# calling the SMME models into a new model called SMME_general
SMME_general_df = pd.DataFrame()
# keep only the rows with the models in SMME_models that matches with the keys in SMME_models
del SMME_models_general['metrics']
for region, seasons in SMME_models_general.items():
    for season, models in seasons.items():
        temp = merged_csv_data[(merged_csv_data['model'].isin(models)) & (merged_csv_data['region'] == region) & (merged_csv_data['season'] == season)]
        SMME_general_df = pd.concat([SMME_general_df, temp], ignore_index=True)

# averaging among the models and renaming to SMME_general
SMME_general_df = SMME_general_df[['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time', 'predicted_precip', 'precip']]\
                .groupby(['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time'])[['predicted_precip', 'precip']].mean().reset_index()
SMME_general_df['model'] = 'SMME_general'

# Saving the SMME_general_df to a csv file
grouped_general = SMME_general_df.groupby('region')
for region, region_df in grouped_general:
    region_name = str(region)
    model = 'SMME_general'
    filename = os.path.join(seasonal_data_save_path, f'{region_name}_{model}_merged_seasonal.csv')
    region_df.to_csv(filename)

# SMME model that only uses short lead metrics (0-1 month lead time)

In [43]:
# keeping short months
potential_skill_short = keep_months_of_prediction(potential_skill, 'short').drop(columns=['lead_time'])

# merging potential skill with an and bn
merged_metrics_short = potential_skill_short.merge(an_bn, on=['region', 'model', 'season', 'month_of_prediction'], how='inner')

# taking the mean of the month of prediction for each region and model and season
merged_metrics_short = merged_metrics_short.groupby(['region', 'model', 'season'])[['potential_skill', 'an_agreement', 'bn_agreement']].mean().reset_index().dropna()

# keep models with potential skill > potential_skill_cutoff or (an_agreement > an_cutoff and bn_agreement > bn_cutoff)
merged_metrics_short = merged_metrics_short[(merged_metrics_short['potential_skill'] > potential_skill_cutoff) |
                                                ((merged_metrics_short['an_agreement'] > an_cutoff) & (merged_metrics_short['bn_agreement'] > bn_cutoff))]
merged_metrics_short['region_season'] = merged_metrics_short['region'] + " | " + merged_metrics_short['season'] # compile region and season into one column, split by " | "
merged_metrics_short = merged_metrics_short.drop(columns = ['region', 'season'])

In [48]:
# create a nested dictionary where the keys are the regions, the values are lists of models
models_short = merged_metrics_short.groupby('region_season')['model'].apply(list).to_dict()
models_short

# separate the region and season into nested keys
SMME_models_short = {}
SMME_models_short['metrics'] = {
    'potential_skill': potential_skill_cutoff,
    'an_agreement': an_cutoff,
    'bn_agreement': bn_cutoff,
    'SMME_criteria': f'(potential_skill > {potential_skill_cutoff}) OR (an AND bn) > {an_cutoff}'
}
for region_season, model_list in models_short.items():
    region, season = region_season.split(" | ")
    if region not in SMME_models_short:
        SMME_models_short[region] = {}
    SMME_models_short[region][season] = model_list

# create a text file of the SMME models chosen
with open(os.path.join(SMME_text_save_path, 'SMME_models_short.txt'), 'w') as f:
    f.write("SMME Short Lead Models\n")
    for region, seasons in SMME_models_short.items():
        f.write(f"{region}:\n")
        for season, models in seasons.items():
            f.write(f"  {season}: {models}\n")

In [52]:
# calling the SMME models into a new model called SMME_short
SMME_short_df = pd.DataFrame()
# keep only the rows with the models in SMME_models that matches with the keys in SMME_models
del SMME_models_short['metrics']
for region, seasons in SMME_models_short.items():
    for season, models in seasons.items():
        temp = merged_csv_data[(merged_csv_data['model'].isin(models)) & (merged_csv_data['region'] == region) & (merged_csv_data['season'] == season)]
        SMME_short_df = pd.concat([SMME_short_df, temp], ignore_index=True)

# averaging among the models and renaming to SMME_short
SMME_short_df = SMME_short_df[['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time', 'predicted_precip', 'precip']]\
                .groupby(['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time'])[['predicted_precip', 'precip']].mean().reset_index()
SMME_short_df['model'] = 'SMME_short'

# Saving the SMME_short_df to a csv file
grouped_short = SMME_short_df.groupby('region')
for region, region_df in grouped_short:
    region_name = str(region)
    model = 'SMME_short'
    filename = os.path.join(seasonal_data_save_path, f'{region_name}_{model}_merged_seasonal.csv')
    region_df.to_csv(filename)

# SMME model that only uses medium lead metrics (2-3 month lead time)

In [54]:
# keeping medium months
potential_skill_medium = keep_months_of_prediction(potential_skill, 'medium').drop(columns=['lead_time'])

# merging potential skill with an and bn
merged_metrics_medium = potential_skill_medium.merge(an_bn, on=['region', 'model', 'season', 'month_of_prediction'], how='inner')

# taking the mean of the month of prediction for each region and model and season
merged_metrics_medium = merged_metrics_medium.groupby(['region', 'model', 'season'])[['potential_skill', 'an_agreement', 'bn_agreement']].mean().reset_index().dropna()

# keep models with potential skill > potential_skill_cutoff or (an_agreement > an_cutoff and bn_agreement > bn_cutoff)
merged_metrics_medium = merged_metrics_medium[(merged_metrics_medium['potential_skill'] > potential_skill_cutoff) |
                                                ((merged_metrics_medium['an_agreement'] > an_cutoff) & (merged_metrics_medium['bn_agreement'] > bn_cutoff))]
merged_metrics_medium['region_season'] = merged_metrics_medium['region'] + " | " + merged_metrics_medium['season'] # compile region and season into one column, split by " | "
merged_metrics_medium = merged_metrics_medium.drop(columns = ['region', 'season'])

In [56]:
# create a nested dictionary where the keys are the regions, the values are lists of models
models_medium = merged_metrics_medium.groupby('region_season')['model'].apply(list).to_dict()
models_medium

# separate the region and season into nested keys
SMME_models_medium = {}
SMME_models_medium['metrics'] = {
    'potential_skill': potential_skill_cutoff,
    'an_agreement': an_cutoff,
    'bn_agreement': bn_cutoff,
    'SMME_criteria': f'(potential_skill > {potential_skill_cutoff}) OR (an AND bn) > {an_cutoff}'
}
for region_season, model_list in models_medium.items():
    region, season = region_season.split(" | ")
    if region not in SMME_models_medium:
        SMME_models_medium[region] = {}
    SMME_models_medium[region][season] = model_list

# create a text file of the SMME models chosen
with open(os.path.join(SMME_text_save_path, 'SMME_models_medium.txt'), 'w') as f:
    f.write("SMME Medium Lead Models\n")
    for region, seasons in SMME_models_medium.items():
        f.write(f"{region}:\n")
        for season, models in seasons.items():
            f.write(f"  {season}: {models}\n")

In [57]:
# calling the SMME models into a new model called SMME_medium
SMME_medium_df = pd.DataFrame()
# keep only the rows with the models in SMME_models that matches with the keys in SMME_models
del SMME_models_medium['metrics']
for region, seasons in SMME_models_medium.items():
    for season, models in seasons.items():
        temp = merged_csv_data[(merged_csv_data['model'].isin(models)) & (merged_csv_data['region'] == region) & (merged_csv_data['season'] == season)]
        SMME_medium_df = pd.concat([SMME_medium_df, temp], ignore_index=True)

# averaging among the models and renaming to SMME_medium
SMME_medium_df = SMME_medium_df[['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time', 'predicted_precip', 'precip']]\
                .groupby(['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time'])[['predicted_precip', 'precip']].mean().reset_index()
SMME_medium_df['model'] = 'SMME_medium'

# Saving the SMME_medium_df to a csv file
grouped_medium = SMME_medium_df.groupby('region')
for region, region_df in grouped_medium:
    region_name = str(region)
    model = 'SMME_medium'
    filename = os.path.join(seasonal_data_save_path, f'{region_name}_{model}_merged_seasonal.csv')
    region_df.to_csv(filename)

# SMME model that only uses long lead metrics (4-6 month lead time)

In [58]:
# keeping long months
potential_skill_long = keep_months_of_prediction(potential_skill, 'long').drop(columns=['lead_time'])

# merging potential skill with an and bn
merged_metrics_long = potential_skill_long.merge(an_bn, on=['region', 'model', 'season', 'month_of_prediction'], how='inner')

# taking the mean of the month of prediction for each region and model and season
merged_metrics_long = merged_metrics_long.groupby(['region', 'model', 'season'])[['potential_skill', 'an_agreement', 'bn_agreement']].mean().reset_index().dropna()

# keep models with potential skill > potential_skill_cutoff or (an_agreement > an_cutoff and bn_agreement > bn_cutoff)
merged_metrics_long = merged_metrics_long[(merged_metrics_long['potential_skill'] > potential_skill_cutoff) |
                                                ((merged_metrics_long['an_agreement'] > an_cutoff) & (merged_metrics_long['bn_agreement'] > bn_cutoff))]
merged_metrics_long['region_season'] = merged_metrics_long['region'] + " | " + merged_metrics_long['season'] # compile region and season into one column, split by " | "
merged_metrics_long = merged_metrics_long.drop(columns = ['region', 'season'])

In [63]:
# create a nested dictionary where the keys are the regions, the values are lists of models
models_long = merged_metrics_long.groupby('region_season')['model'].apply(list).to_dict()
models_long

# separate the region and season into nested keys
SMME_models_long = {}
SMME_models_long['metrics'] = {
    'potential_skill': potential_skill_cutoff,
    'an_agreement': an_cutoff,
    'bn_agreement': bn_cutoff,
     'SMME_criteria': f'(potential_skill > {potential_skill_cutoff}) OR (an AND bn) > {an_cutoff}'
}
for region_season, model_list in models_long.items():
    region, season = region_season.split(" | ")
    if region not in SMME_models_long:
        SMME_models_long[region] = {}
    SMME_models_long[region][season] = model_list

# create a text file of the SMME models chosen
with open(os.path.join(SMME_text_save_path, 'SMME_models_long.txt'), 'w') as f:
    f.write("SMME Long Lead Models\n")
    for region, seasons in SMME_models_long.items():
        f.write(f"{region}:\n")
        for season, models in seasons.items():
            f.write(f"  {season}: {models}\n")

In [64]:
# calling the SMME models into a new model called SMME_long
SMME_long_df = pd.DataFrame()
# keep only the rows with the models in SMME_models that matches with the keys in SMME_models
del SMME_models_long['metrics']
for region, seasons in SMME_models_long.items():
    for season, models in seasons.items():
        temp = merged_csv_data[(merged_csv_data['model'].isin(models)) & (merged_csv_data['region'] == region) & (merged_csv_data['season'] == season)]
        SMME_long_df = pd.concat([SMME_long_df, temp], ignore_index=True)

# averaging among the models and renaming to SMME_long
SMME_long_df = SMME_long_df[['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time', 'predicted_precip', 'precip']]\
                .groupby(['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time'])[['predicted_precip', 'precip']].mean().reset_index()
SMME_long_df['model'] = 'SMME_long'

# Saving the SMME_long_df to a csv file
grouped_long = SMME_long_df.groupby('region')
for region, region_df in grouped_long:
    region_name = str(region)
    model = 'SMME_long'
    filename = os.path.join(seasonal_data_save_path, f'{region_name}_{model}_merged_seasonal.csv')
    region_df.to_csv(filename)